# Interpolation to query points in parallel

In this example, we will show the workflow that we generally take when operating files that eventually need an interpolation operation.

In example `4.6 - interpolating_file_sequence`, we will later show how to quickly interpolate files from disk. However, in those cases, IO for probes is in serial, reading in `rank 0` and scattering the data.

We have found that while very useful, sometimes the amount of data and operations to make call for a different approach. We hope that this example can help you get started quickly in such scenarios.

> The goal of this example is to showcase a workflow where **every step** is in parallel

## SEM mesh and query points

For interpolation, we allways require a spectral element mesh (SEM) that contains the original data, and we also need a set of query points to interpolate to.

We recommend that both of these are files in disk. For query points, it is a good idea to have a prepared `hdf5` file. You can see how we create ours in example `4.5 - structured_mesh`. For this example we use the previously created points in `../data/points.hdf5`

The SEM mesh corresponds to a cylindrical domain located at: `../data/rbc0.f00001`

> We recommend that you set up you mesh such that you can partition it across ranks based on `axis 0` from its shape
> this is not a hard requirement if you only want to use `hdf5` files. But it is needed if you want `vtkhdf`

In [1]:
sem_mesh_fname = '../data/rbc0.f00001'
query_points_fname = '../data/points.hdf5'

we also need to define a file sequence of data that will be interpolated. In this case, we will just have one file in our sequence:

In [2]:
file_sequence = ['../data/rbc0.f00001']

## Read mesh/query data

Now we start with the io

In [3]:
# Import required modules
from mpi4py import MPI #equivalent to the use of MPI_init() in C
import numpy as np

# Get mpi info
comm = MPI.COMM_WORLD

For the query points, we will use our wrapper. For the nek-like type we will use the dedicated reader. In reality, the wrapper can be used in both cases.

Please see example `1.1 - datatypes_io` to know more on ways to perform IO

In [4]:
# Read the query points
from pysemtools.io.wrappers import read_data
query_mesh = read_data(comm, query_points_fname, keys=['x', 'y', 'z'], parallel_io=True, distributed_axis=0)

# Read the SEM mesh
from pysemtools.datatypes import Mesh, FieldRegistry, Coef
from pysemtools.io.ppymech import pynekread
msh = Mesh(comm)
pynekread(sem_mesh_fname, comm, msh=msh)

# Optionally build Coefs if you need derivatives, etc. Be mindful that this requires a bit of memory
coef = Coef(msh, comm)

2026-03-04 22:07:55,819  HDF5File              INFO      ../data/points.hdf5 opened - mode r - serial
2026-03-04 22:07:55,823  HDF5File              INFO      ../data/points.hdf5 closed - Elapsed time: 0.004483641s
2026-03-04 22:07:55,824  pynekread             INFO      Reading file: ../data/rbc0.f00001
2026-03-04 22:07:55,843  Mesh                  INFO      Mesh object initialized from coordinates with type: float64 - Elapsed time: 0.010797221s
2026-03-04 22:07:55,844  pynekread             INFO      File successfully read - Elapsed time: 0.020370785000000002s
2026-03-04 22:07:55,887  Coef                  INFO      Coef object initialized - dtype: float64 - Elapsed time: 0.04161013999999999s


Note that if running this on a python script on multiple ranks, both of these data sets will already be distributed among ranks at this stage. Here we read the `x`, `y`, and `z` keys of the hdf5 file that contains the query points

**Important** the query points need to be given as an array with shape `(n_points, 3)` and right now our query points are 3 different arrays with shape `(nx, ny, nz)`. So we need to do a preprocessing step:

In [5]:
xyz = [query_mesh["x"].flatten(), query_mesh["y"].flatten() , query_mesh["z"].flatten() ]
xyz = np.ascontiguousarray(np.array(xyz).T)

## Operate and interpolate data

We can now do any operations we need in our file sequence.

We leave a code block section empty, where you could include any operation you need.

> Distributed axis here needs to be the same used to read the data, otherwise errors might appear.
> **Importat:** Since we output in `vtkhdf`, we **need** distributed axis `0`

Note that after the fields are interpolated into the query points, we reshape the data back to the original shape.

In [6]:
from pysemtools.interpolation import Probes
from pysemtools.io.wrappers import write_data
for fi, fname in enumerate(file_sequence):

    # ===========================
    # Initialize the interpolator
    # ===========================
    if fi == 0: # Only at step 0
        probes = Probes(comm, probes = xyz, msh = msh, 
                        point_interpolator_type='multiple_point_legendre_numpy', 
                        max_pts=256, find_points_comm_pattern='point_to_point',
                        write_coords=False, output_fname='out.hdf5')
        probes.log.write('info', '==============================================================================')

    # =============================
    # Read current file in sequence
    # =============================
    fld = FieldRegistry(comm)
    pynekread(fname, comm, fld=fld)

    # ================================
    # Do operations that you need here
    # ================================


    # ===============================
    # Interpolate to the query points
    # ===============================
    field_list = [fld.registry[key] for key in fld.registry.keys()]
    field_names = [key for key in fld.registry.keys()]
    probes.interpolate_from_field_list(fld.t, field_list, comm, write_data=False, field_names=field_names)
    
    # ====================================
    # Output the data in a format you need
    # ====================================
    fname_out = f'interpolated_{fi:05d}.vtkhdf'
    output_data = {}
    for i, key in enumerate(field_names):
        output_data[key] = probes.interpolated_fields[:, i+1].reshape(query_mesh['x'].shape).astype(np.double) 
    write_data(comm, fname_out, data_dict=output_data, parallel_io=True, distributed_axis=0, msh=[query_mesh[key] for key in query_mesh.keys()])
    

2026-03-04 22:07:56,079  Probes                INFO      Initializing Probes object
2026-03-04 22:07:56,083  Probes                INFO      Query points (probes) read/assigned - Elapsed time: 4.936000000010932e-06s
2026-03-04 22:07:56,093  Probes                INFO      Interpolator initialized in all ranks - Elapsed time: 0.009478862000000032s
2026-03-04 22:07:56,099  Interpolator          INFO      Elapsed time: 0.004858552999999988s
2026-03-04 22:07:56,099  Interpolator          INFO      Finding points - start
2026-03-04 22:07:56,238  Interpolator          INFO      Processing data in multiple ranks. Reporting progress only for rank 0
2026-03-04 22:07:56,239  Interpolator          INFO      Processing batch: 1 out of 1
2026-03-04 22:07:59,467  Probes                INFO      Finding points finalized in all ranks - Elapsed time: 3.368195352s
2026-03-04 22:07:59,472  Interpolator          INFO      Elapsed time: 0.003914360000000006s
2026-03-04 22:07:59,476  Probes                I